<!-- VIDEO: 1 / create_agent — 한 줄로 시작하는 ReAct 에이전트 -->

# 05. Agents — `create_agent`

> **학습 목표**
> 1. **LangChain v1.0의 `create_agent`** 를 사용해 ReAct 에이전트를 한 줄로 구성한다.
> 2. **ReAct 루프**(Reason → Act → Observe)의 흐름을 시각화하고, 04번에서 직접 구현한 수동 루프와 비교한다.
> 3. `stream()`을 활용해 에이전트의 사고 과정을 실시간으로 관찰한다.
> 4. **`MemorySaver` 체크포인터**로 멀티턴 대화를 구현한다.
> 5. `response_format`(`ToolStrategy` / `ProviderStrategy`)을 사용한 구조화 출력을 적용한다.
> 6. `@dynamic_prompt` 미들웨어로 컨텍스트(사용자 권한 등)에 따른 동적 프롬프트 분기를 구현한다.
>
> **선수 지식**: 04번 노트북(수동 tool-calling 루프).

---

## LangChain v1.0의 변경점

LangChain 0.x 시점에는 에이전트 생성 API가 분산되어 있어 선택에 혼란이 있었습니다.

- `AgentExecutor` + `create_react_agent` / `create_tool_calling_agent`
- `langgraph.prebuilt.create_react_agent`

v1.0부터는 `langchain.agents.create_agent` 단일 함수로 통합되었습니다. 내부적으로 LangGraph 위에서 동작하므로 **체크포인팅, 스트리밍, HITL(Human-in-the-Loop), 상태 시간여행(time travel)** 기능이 기본으로 제공됩니다.

> **04번 수동 루프와의 비교**: 04번에서 직접 구현한 `run_tool_loop`는 약 30줄의 함수였습니다. 본 노트북에서는 `create_agent(model=llm, tools=tools, system_prompt="...")` 단일 호출로 동일한 기능에 더해 메모리, 스트리밍, 가드레일까지 함께 제공받습니다.

---

## 1. 환경 설정

이전 노트북과 동일한 LLM 초기화 패턴을 사용합니다. 보유한 키에 따라 자동으로 제공자가 분기됩니다.

In [2]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.checkpoint.memory import MemorySaver
import os, json

TAVILY_OK = bool(os.getenv("TAVILY_API_KEY"))

# ✅ Option 1: Google Gemini 2.5 Flash-Lite (무료) — thinking_budget=0 으로 추론 토큰 비활성화
# llm = init_chat_model("google_genai:gemini-2.5-flash-lite", temperature=0,
#                        model_kwargs={"thinking_budget": 0})

# ✅ Option 2: Groq Qwen3 32B (무료) — reasoning_effort="none" 으로 <think> 토큰 비활성화
# llm = init_chat_model("groq:qwen/qwen3-32b", temperature=0, reasoning_effort="none")

# ✅ Option 3: Ollama Gemma 4 E4B (로컬) — reasoning=False 으로 <think> 토큰 비활성화
llm = init_chat_model("ollama:gemma4:e4b", temperature=0, reasoning=False)

# ⚙️ Option 4: OpenAI (유료) — reasoning 토큰 없음, 추가 파라미터 불필요
# llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)

print(f"✅ ready (Tavily: {'on' if TAVILY_OK else 'fallback'})")

✅ ready (Tavily: on)


## 2. 에이전트와 체인의 비교

| 구분 | 체인 (LCEL) | 에이전트 |
|---|---|---|
| 처리 흐름 | **고정** (예: prompt → llm → parser) | **동적** — 모델이 도구 호출 결과를 보고 다음 행동을 결정 |
| 제어 주체 | 개발자가 처리 순서를 코딩 | 모델이 상황에 따라 자율 결정 |
| 도구 사용 | 수동 호출 필요 | 자동 (ReAct 루프) |
| 적합한 사례 | 단순 변환 파이프라인 | 복합 질의, 외부 시스템 조회 필요 작업 |

### ReAct(Reason → Act → Observe) 패턴

```mermaid
flowchart TD
    IN(["💬 사용자 질문<br/>HumanMessage"]):::input
    R["🧠 REASON<br/>LLM이 필요한 도구를 판단"]:::reason
    A["⚙️ ACT<br/>호출 측 코드가 도구 실행<br/>tool.invoke#40;args#41;"]:::act
    O["👁️ OBSERVE<br/>LLM이 결과를 바탕으로<br/>다음 행동 결정"]:::observe
    CHK{"추가 도구<br/>호출 필요?"}:::decision
    OUT(["✅ 최종 답변<br/>AIMessage"]):::output

    IN  --> R
    R   -->|"tool_calls"| A
    A   -->|"ToolMessage"| O
    O   --> CHK
    CHK -->|"Yes"| R
    CHK -->|"No"| OUT

    classDef input    fill:#4f46e5,stroke:#3730a3,color:#fff
    classDef reason   fill:#0f172a,stroke:#6366f1,color:#a5b4fc
    classDef act      fill:#0f172a,stroke:#f59e0b,color:#fcd34d
    classDef observe  fill:#0f172a,stroke:#10b981,color:#6ee7b7
    classDef decision fill:#1e293b,stroke:#94a3b8,color:#e2e8f0
    classDef output   fill:#059669,stroke:#047857,color:#fff
```

04번 노트북에서 직접 구현한 `run_tool_loop`가 정확히 이 흐름을 따랐습니다. `create_agent`는 동일한 루프를 단일 호출로 대체하면서, 메모리·스트리밍·가드레일 기능을 함께 제공합니다.

### ReAct 4단계 요약

1. **Reason**: 모델이 사용자 질문을 분석하고 필요한 도구를 판단합니다.
2. **Act**: `tool_calls`로 도구 호출을 요청합니다.
3. **Observe**: 도구 실행 결과를 `ToolMessage`로 받아 다음 판단의 근거로 활용합니다.
4. 추가 작업이 필요하면 1~3을 반복하고, 그렇지 않으면 최종 답변을 생성합니다.

## 3. `create_agent` 시그니처

```python
create_agent(
    model,                    # str 또는 BaseChatModel (예: "gpt-4.1-mini" 또는 llm 인스턴스)
    tools=[...],              # @tool 함수, 일반 함수, 또는 dict (빌트인 프로바이더 도구)
    *,
    system_prompt=None,       # str 또는 None. v0.x의 prompt= 가 v1.0에서 system_prompt= 로 변경됨
    middleware=(),            # AgentMiddleware 리스트
    response_format=None,     # ToolStrategy(Schema) 또는 ProviderStrategy(Schema)
    state_schema=None,        # 사용자 정의 TypedDict 상태
    context_schema=None,      # 런타임 컨텍스트 dataclass
    checkpointer=None,        # MemorySaver 등 — 대화 이력 저장
    store=None,               # 장기 메모리 스토어
    interrupt_before=None,    # HITL
    interrupt_after=None,     # HITL
    debug=False,
    name=None,
    cache=None,
)
```

## 4. 기본 에이전트 구성

04번 노트북의 도구 일부를 재사용해, 단일 호출로 에이전트를 구성합니다.

In [3]:
from datetime import date, datetime

@tool
def get_schedule(date: str) -> list[dict]:
    """주어진 날짜(YYYY-MM-DD)의 사내 일정을 조회합니다."""
    fake_db = {
        "2026-04-15": [
            {"time": "10:00", "title": "주간 스탠드업", "room": "A"},
            {"time": "14:00", "title": "고객 데모", "room": "Zoom"},
        ],
        "2026-04-16": [{"time": "09:30", "title": "OKR 리뷰", "room": "대회의실"}],
    }
    return fake_db.get(date, [])


@tool
def calc_leave_days(start: str, end: str) -> dict:
    """두 날짜 사이의 영업일(주말 제외)을 계산합니다."""
    d1 = datetime.strptime(start, "%Y-%m-%d").date()
    d2 = datetime.strptime(end, "%Y-%m-%d").date()
    days = sum(1 for i in range((d2 - d1).days + 1)
               if (d1.fromordinal(d1.toordinal() + i)).weekday() < 5)
    return {"start": start, "end": end, "business_days": days}


@tool
def get_leave_balance(employee_id: str) -> dict:
    """사원 잔여 연차를 조회합니다."""
    db = {"E12345": 12.5, "E67890": 7.0}
    return {"employee_id": employee_id, "remaining_days": db.get(employee_id, 10.0)}


# 웹검색 (실제 또는 폴백)
if TAVILY_OK:
    from langchain_tavily import TavilySearch
    web_search = TavilySearch(max_results=3)
else:
    @tool
    def web_search(query: str) -> list[dict]:
        """웹 검색 (폴백 시뮬레이션)."""
        return [{"title": f"(mock) {query}", "url": "https://example.com",
                 "content": f"mock result for {query}"}]

tools = [get_schedule, calc_leave_days, get_leave_balance, web_search]
print(f"tools: {[t.name for t in tools]}")

tools: ['get_schedule', 'calc_leave_days', 'get_leave_balance', 'tavily_search']


In [4]:
# ⭐ 핵심: 한 줄로 에이전트 생성
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "당신은 회사의 사내 업무 어시스턴트입니다. "
        "일정/휴가/웹검색 도구를 적절히 활용해 사용자 질문에 정확히 답하세요. "
        "날짜 계산이 필요하면 calc_leave_days 도구를 사용하세요."
    ),
)

print(type(agent).__name__)
print("에이전트 준비 완료")

CompiledStateGraph
에이전트 준비 완료


### 에이전트 호출

입력 형식은 메시지 리스트입니다(`{"messages": [...]}`). 출력에는 사용자 메시지부터 모델 응답, 도구 호출 메시지가 순서대로 누적됩니다.

In [5]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "2026년 4월 15일 내 일정을 알려줘."}]
})

# 모든 메시지 확인 (사용자 → 도구 호출 → 도구 결과 → 최종 답변)
for m in result["messages"]:
    kind = type(m).__name__
    if hasattr(m, "tool_calls") and m.tool_calls:
        print(f"[{kind}] tool_calls: {m.tool_calls}")
    elif kind == "ToolMessage":
        print(f"[{kind}] {m.content[:100]}")
    else:
        print(f"[{kind}] {m.content[:200]}")

[HumanMessage] 2026년 4월 15일 내 일정을 알려줘.
[AIMessage] tool_calls: [{'name': 'get_schedule', 'args': {'date': '2026-04-15'}, 'id': 'b50b52ce-4d7d-498b-93c1-623aee8db1a6', 'type': 'tool_call'}]
[ToolMessage] [{"time": "10:00", "title": "주간 스탠드업", "room": "A"}, {"time": "14:00", "title": "고객 데모", "room": "Zo
[AIMessage] 2026년 4월 15일 일정은 다음과 같습니다:

*   **10:00**: 주간 스탠드업 (장소: A)
*   **14:00**: 고객 데모 (장소: Zoom)


최종 답변만 추출하려면 마지막 메시지의 `content`를 사용합니다.

In [6]:
print(result["messages"][-1].content)

2026년 4월 15일 일정은 다음과 같습니다:

*   **10:00**: 주간 스탠드업 (장소: A)
*   **14:00**: 고객 데모 (장소: Zoom)


## 5. 복합 질문 — 다중 도구의 자율 조합

에이전트의 강점은 **여러 도구를 자율적으로 조합**해야 답변 가능한 복합 질문에서 드러납니다. 모델이 처리 순서를 스스로 결정하고, 각 도구의 결과를 다음 단계의 입력으로 활용합니다.

In [7]:
q = (
    "사번 E12345 인 직원이 2026-04-20부터 2026-04-24까지 휴가를 쓰려고 해. "
    "사용 가능한 영업일이 충분한지 계산해서 알려줘."
)
result = agent.invoke({"messages": [{"role": "user", "content": q}]})
print(result["messages"][-1].content)

직원님의 휴가 기간(2026-04-20 ~ 2026-04-24) 동안의 영업일은 **5일**입니다.

사번 E12345 직원의 현재 잔여 연차는 **12.5일**이므로, 해당 휴가 사용에 필요한 영업일보다 충분합니다.


In [8]:
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

사번 E12345 인 직원이 2026-04-20부터 2026-04-24까지 휴가를 쓰려고 해. 사용 가능한 영업일이 충분한지 계산해서 알려줘.
================================== Ai Message ==================================
Tool Calls:
  calc_leave_days (f5ad2a13-de14-44b1-ac2d-eb1bf76d9627)
 Call ID: f5ad2a13-de14-44b1-ac2d-eb1bf76d9627
  Args:
    end: 2026-04-24
    start: 2026-04-20
================================= Tool Message =================================
Name: calc_leave_days

{"start": "2026-04-20", "end": "2026-04-24", "business_days": 5}
================================== Ai Message ==================================
Tool Calls:
  get_leave_balance (5752a01d-9ad1-48ad-8896-82b41cd98254)
 Call ID: 5752a01d-9ad1-48ad-8896-82b41cd98254
  Args:
    employee_id: E12345
================================= Tool Message =================================
Name: get_leave_balance

{"employee_id": "E12345", "remaining_days": 12.5}
================================== 

<!-- VIDEO: 2 / 스트리밍을 통한 에이전트 사고 과정 관찰 -->

## 6. 스트리밍 — 처리 과정 실시간 관찰

`agent.stream(..., stream_mode="values")` 또는 `stream_mode="updates"`를 사용하면 에이전트의 처리 단계를 실시간으로 관찰할 수 있습니다. 디버깅과 사용자 인터페이스의 응답성 향상 모두에 활용됩니다.

In [9]:
print("=" * 60)
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "2026년 4월 16일 일정 알려줘"}]},
    stream_mode="values",
):
    last = chunk["messages"][-1]
    kind = type(last).__name__
    if hasattr(last, "tool_calls") and last.tool_calls:
        print(f"🤔 REASON → ACT: {last.tool_calls[0]['name']}({last.tool_calls[0]['args']})")
    elif kind == "ToolMessage":
        print(f"👁  OBSERVE: {last.content[:100]}")
    elif kind == "AIMessage":
        print(f"💬 ANSWER: {last.content[:200]}")
    elif kind == "HumanMessage":
        print(f"❓ USER: {last.content}")
    print("-" * 60)

❓ USER: 2026년 4월 16일 일정 알려줘
------------------------------------------------------------
🤔 REASON → ACT: get_schedule({'date': '2026-04-16'})
------------------------------------------------------------
👁  OBSERVE: [{"time": "09:30", "title": "OKR 리뷰", "room": "대회의실"}]
------------------------------------------------------------
💬 ANSWER: 2026년 4월 16일 일정은 **09:30에 'OKR 리뷰'**가 **대회의실**에서 예정되어 있습니다.
------------------------------------------------------------


<!-- VIDEO: 3 / 메모리·체크포인트 -->

## 7. 메모리 — `MemorySaver` 체크포인터

`MemorySaver`는 `thread_id` 단위로 대화 이력을 자동 저장하는 체크포인터입니다.

- **`thread_id`**: 대화 세션을 식별하는 키. 동일 `thread_id`로 호출하면 이전 대화 이력이 자동 복원됩니다.
- **자동 저장**: 각 turn 종료 시 메시지 이력이 자동으로 체크포인트에 저장됩니다.
- **세션 격리**: 다른 `thread_id`로 호출하면 별개의 대화 세션으로 처리됩니다.

에이전트 생성 시 `checkpointer=MemorySaver()`를 지정하고, 호출 시 `config={"configurable": {"thread_id": "..."}}`를 전달하면 대화 이력이 `thread_id`별로 자동 관리됩니다.

In [10]:
memory = MemorySaver()
agent_with_memory = create_agent(
    model=llm,
    tools=tools,
    system_prompt="당신은 사내 업무 어시스턴트입니다.",
    checkpointer=memory,
)

cfg = {"configurable": {"thread_id": "session-001"}}

# 첫 번째 턴
r1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "내 사번은 E12345 이고, 이름은 길동이야."}]},
    config=cfg,
)
print("[1] ", r1["messages"][-1].content)
print()

# 두 번째 턴 — 이전 정보를 기억해야 함
r2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "내 잔여 연차가 얼마나 남았지?"}]},
    config=cfg,
)
print("[2] ", r2["messages"][-1].content)

[1]  네, 길동님. 사번은 E12345 이시군요.

제가 어떤 업무를 도와드릴까요? 예를 들어, **남은 연차 조회**, **일정 확인**, 아니면 **다른 정보 검색** 등이 필요하신가요?

[2]  길동님의 현재 잔여 연차는 **12.5일**입니다.

혹시 연차 사용 계획이나 다른 궁금한 점이 있으신가요?


### 다른 `thread_id`는 독립적인 대화 세션

새로운 `thread_id`로 호출하면 이전 대화 이력이 공유되지 않으며, 모델은 이전 맥락을 인지하지 못합니다.

In [11]:
cfg2 = {"configurable": {"thread_id": "session-002"}}
r3 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "내 사번 기억나?"}]},
    config=cfg2,
)
# 새 세션이므로 사번을 모를 것
print("[new session]", r3["messages"][-1].content)

[new session] 죄송하지만, 저는 사용자의 개인 식별 정보(사번 포함)를 기억하거나 저장할 수 없습니다. 저는 대화가 끝날 때마다 이전 대화 내용을 초기화하는 방식으로 작동하기 때문에, 사번을 알려주셔도 다음 대화에서는 알 수 없습니다.

혹시 사번이 필요하신 업무가 있으신가요? 예를 들어, 연차 조회나 일정 확인 등 어떤 도움이 필요하신지 말씀해주시면, 필요한 정보를 요청드리겠습니다.


<!-- VIDEO: 4 / 구조화 출력 — ToolStrategy / ProviderStrategy -->

## 8. 구조화 출력 — `ToolStrategy` / `ProviderStrategy`

에이전트의 최종 답변을 **Pydantic 스키마에 맞춘 구조화 형식**으로 받고자 할 때 `response_format` 파라미터를 사용합니다. v1.0의 `create_agent`는 별도의 추가 LLM 호출 없이 기본 처리 루프 안에서 구조화 출력을 생성합니다.

| 전략 | 동작 방식 | 호환성 |
|---|---|---|
| `ToolStrategy(Schema)` | 스키마를 도구 형태로 모델에 노출하여, 모델이 해당 도구를 호출하는 방식으로 응답 | 도구 호출을 지원하는 모든 모델 |
| `ProviderStrategy(Schema)` | OpenAI, Anthropic 등 제공자의 **네이티브 구조화 출력 API** 활용 | 일부 최신 모델 (gpt-4.1, gpt-4o 계열 등) |

<br>

> **자동 폴백**: `response_format=MySchema`처럼 스키마만 전달하면, 모델이 네이티브 구조화 출력을 지원할 경우 `ProviderStrategy`로, 그렇지 않으면 `ToolStrategy`로 자동 선택됩니다.

결과는 `result["structured_response"]`에서 추출합니다.

### 8-1. `ToolStrategy` — 도구 호출과 구조화 출력의 결합

`get_leave_balance`, `calc_leave_days` 등의 도구를 자율적으로 호출하면서, 최종 답변은 `LeaveReport` 스키마 형식으로 반환받습니다.

In [12]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy, ProviderStrategy


class LeaveReport(BaseModel):
    """휴가 신청 사전 점검 결과."""
    employee_id: str = Field(description="사번")
    requested_business_days: int = Field(description="신청 영업일 수")
    remaining_days: float = Field(description="현재 잔여 연차")
    can_approve: bool = Field(description="잔여 연차로 승인 가능한지 여부")
    summary: str = Field(description="한국어로 2~3문장 요약")


agent_struct = create_agent(
    model=llm,
    tools=[get_leave_balance, calc_leave_days],
    system_prompt=(
        "당신은 휴가 심사 보조원입니다. "
        "필요한 도구를 호출해 정보를 모은 뒤, LeaveReport 스키마에 맞춰 답하세요."
    ),
    response_format=ToolStrategy(LeaveReport),
)

result = agent_struct.invoke({
    "messages": [{
        "role": "user",
        "content": "사번 E12345 가 2026-04-20부터 2026-04-24까지 휴가 쓰려고 해. 승인 가능한지 보고해줘.",
    }]
})

report: LeaveReport = result["structured_response"]
print("type:", type(report).__name__)
print(report.model_dump_json(indent=2))

type: LeaveReport
{
  "employee_id": "E12345",
  "requested_business_days": 5,
  "remaining_days": 12.5,
  "can_approve": true,
  "summary": "사번 E12345님의 2026년 4월 20일부터 4월 24일까지의 휴가 신청 건입니다. 총 5일의 영업일이 필요하며, 현재 잔여 연차는 12.5일로 충분하여 승인 가능합니다."
}


### 8-2. `ProviderStrategy` — 제공자 네이티브 구조화 출력

내부적으로 OpenAI의 `response_format={"type": "json_schema", ...}` 등 제공자 네이티브 API를 사용합니다. 모델이 해당 기능을 지원하는 경우 일반적으로 더 안정적이고 빠르게 동작하며, 도구 슬롯을 차지하지 않습니다.

아래 예시는 도구 없이 단순 정보 추출만 수행하는 경우입니다.

In [13]:
class ContactInfo(BaseModel):
    """고객 문의에서 추출한 연락처 정보."""
    name: str = Field(description="담당자 이름")
    email: str = Field(description="이메일 주소")
    phone: str = Field(description="전화번호 (하이픈 포함)")
    company: str | None = Field(default=None, description="회사명 (언급된 경우)")


agent_extract = create_agent(
    model=llm,
    tools=[],                                       # 도구 불필요 — 순수 추출 작업
    system_prompt="고객 메시지에서 연락처를 정확히 추출하세요.",
    response_format=ProviderStrategy(ContactInfo),
)

msg = (
    "안녕하세요, 교육팀의 고길동 과장입니다. "
    "문의 답변은 gd.ko@gdko.co.kr 또는 010-1234-5678 로 부탁드립니다."
)

result = agent_extract.invoke({"messages": [{"role": "user", "content": msg}]})
contact: ContactInfo = result["structured_response"]
print(contact.model_dump_json(indent=2))

{
  "name": "고길동",
  "email": "gd.ko@gdko.co.kr",
  "phone": "010-1234-5678",
  "company": null
}


### 8-3. 스키마만 전달하기 — 자동 전략 선택

v1.0에서는 `response_format=Schema`처럼 스키마만 전달해도 동작합니다. 모델이 네이티브 구조화 출력을 지원하면 `ProviderStrategy`, 그렇지 않으면 `ToolStrategy`로 **자동 선택**됩니다. 어떤 전략이 사용되는지 직접 관리할 필요가 없는 가장 간편한 방식입니다.

In [14]:
class ScheduleSummary(BaseModel):
    """하루 일정 요약."""
    date: str = Field(description="조회한 날짜 (YYYY-MM-DD)")
    total: int = Field(description="일정 개수")
    titles: list[str] = Field(description="일정 제목 리스트")


# 스키마만 전달 → 내부에서 적절한 전략 자동 선택
agent_auto = create_agent(
    model=llm,
    tools=[get_schedule],
    system_prompt="요청된 날짜의 일정을 조회해 ScheduleSummary 로 답하세요.",
    response_format=ScheduleSummary,
)

result = agent_auto.invoke({
    "messages": [{"role": "user", "content": "2026-04-15 일정 요약해줘"}]
})

summary: ScheduleSummary = result["structured_response"]
print(summary)
print("titles:", summary.titles)

date='2026-04-15' total=2 titles=['주간 스탠드업', '고객 데모']
titles: ['주간 스탠드업', '고객 데모']


### 8-4. 전략 선택 가이드

| 상황 | 권장 전략 |
|---|---|
| 프로토타이핑, 모델 교체 가능성을 고려해야 하는 경우 | `response_format=Schema` (자동 선택) |
| 안정성을 최우선으로 하며 OpenAI/Anthropic 최신 모델을 사용하는 경우 | `ProviderStrategy(Schema)` |
| 구형 또는 오픈소스 모델을 포함해 여러 제공자에서 동일하게 동작해야 하는 경우 | `ToolStrategy(Schema)` |
| 도구 호출과 구조화 출력을 동일 루프 내에서 처리해야 하는 경우 | 두 전략 모두 가능 (8-1 참고) |

<br>

> 일부 제공자는 도구 호출과 `ProviderStrategy`를 동시에 사용할 수 없으며, 이 경우 내부적으로 `ToolStrategy`로 폴백됩니다. 사용 모델별 문서를 확인하시기 바랍니다.

<!-- VIDEO: 5 / @dynamic_prompt — 컨텍스트 기반 프롬프트 분기 -->

## 9. 동적 프롬프트 — `@dynamic_prompt` 미들웨어

런타임 컨텍스트(예: 사용자 권한, 언어 설정)에 따라 시스템 프롬프트를 다르게 적용해야 하는 경우, v1.0에서는 `middleware` 파라미터에 `@dynamic_prompt` 데코레이트된 함수를 등록하여 처리합니다.

> **v0.x와의 차이**: 0.x에서는 `prompt=callable` 형태로 함수를 직접 전달했으나, v1.0에서는 반드시 `@dynamic_prompt` 데코레이터로 감싼 후 `middleware=[...]`에 등록해야 합니다.

In [15]:
from dataclasses import dataclass
from langchain.agents.middleware import dynamic_prompt
# ⚠️ ModelRequest 는 일부 버전에서 .types 서브모듈에 위치 — 두 경로 중 동작하는 것 사용
try:
    from langchain.agents.middleware import ModelRequest
except ImportError:
    from langchain.agents.middleware.types import ModelRequest


@dataclass
class UserContext:
    user_level: str = "user"       # 'user' 또는 'admin'
    user_name: str = ""
    employee_id: str = ""           # 도구 호출에 바로 사용


@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    ctx: UserContext = request.runtime.context
    base = "당신은 사내 업무 어시스턴트입니다."

    if ctx.user_level == "admin":
        return (
            f"{base}\n"
            f"[호출자] {ctx.user_name} / 권한: **관리자** / 사번: {ctx.employee_id}\n"
            "[답변 규칙]\n"
            "- 반드시 '📊 [관리자 리포트]' 로 시작합니다.\n"
            "- 다음 항목을 **모두** 글머리표(•)로 포함합니다:\n"
            "   • 사번(raw 그대로)\n"
            "   • 잔여 연차(소수점 포함 정확한 숫자)\n"
            "   • 내부 정책 수치: 연차 1일 = 근무 8시간, 연간 부여 15일, 이월 한도 5일\n"
            "   • 타 직원 대비 상대 수준 (추정 코멘트 1줄)\n"
            "- 경어체는 유지하되 수치/PII 를 가리지 말고 투명하게 공개합니다.\n"
            "- 도구 호출 시 employee_id 는 호출자 사번을 그대로 사용하세요."
        )
    else:
        return (
            f"{base}\n"
            f"[호출자] {ctx.user_name} / 권한: 일반 직원 / 사번: {ctx.employee_id}\n"
            "[답변 규칙]\n"
            "- 반드시 '🙂 [안내]' 로 시작하며 **한두 문장** 으로만 답합니다.\n"
            "- 사번·정확한 소수점·타 직원 정보·내부 정책 수치는 **절대 공개 금지**.\n"
            "- 잔여 연차는 '약 N일 정도' 처럼 **반올림한 자연어** 로만 표현합니다.\n"
            "- 팀 평균/타 직원 비교 요청은 정중히 거절하세요.\n"
            "- 도구 호출 시 employee_id 는 호출자 사번을 그대로 사용하세요."
        )


agent_dyn = create_agent(
    model=llm,
    tools=tools,
    middleware=[personalized_prompt],     # system_prompt 는 미들웨어가 제공
    context_schema=UserContext,
)

# 같은 질문 — "내 연차 + 팀원 평균 + 정책 상세" 를 한 번에 요청 → 권한별 응답이 확연히 갈림
question = (
    "제 잔여 연차를 알려주시고, 팀원 평균 대비 어느 정도인지, "
    "그리고 이월 한도 같은 내부 정책 수치도 함께 설명해 주세요."
)

# 일반 사용자 호출
r_user = agent_dyn.invoke(
    {"messages": [{"role": "user", "content": question}]},
    context=UserContext(user_level="user", user_name="고길동", employee_id="E67890"),
)
print("[USER]")
print(r_user["messages"][-1].content)
print()
print("-" * 60)
print()

# 관리자 호출
r_admin = agent_dyn.invoke(
    {"messages": [{"role": "user", "content": question}]},
    context=UserContext(user_level="admin", user_name="김부장", employee_id="E12345"),
)
print("[ADMIN]")
print(r_admin["messages"][-1].content)

[USER]
🙂 [안내] 잔여 연차는 조회해 드릴 수 있지만, 팀 평균이나 내부 정책 수치와 같은 개인 정보는 안내해 드리기 어렵습니다.

------------------------------------------------------------

[ADMIN]
📊 [관리자 리포트]
요청하신 사원님의 잔여 연차 정보 및 내부 정책 수치를 안내해 드리겠습니다.

먼저, 사원님의 잔여 연차를 조회하기 위해 사번을 알려주시거나, 제가 임의로 조회할 수 있는 사번을 지정해 주셔야 합니다. 현재는 사번 정보가 없어 정확한 조회가 어렵습니다.

**만약 사번을 알려주시면, 아래와 같은 형식으로 정보를 제공해 드릴 수 있습니다.**

*   **사번**: [사번]
*   **잔여 연차**: [조회된 정확한 숫자]일
*   **내부 정책 수치**:
    *   연차 1일 = 근무 8시간
    *   연간 부여 일수: 15일
    *   연차 이월 한도: 5일
*   **타 직원 대비 상대 수준**: (조회된 데이터를 바탕으로 추정 코멘트 1줄)

**참고:**
1.  **잔여 연차 조회**: `get_leave_balance` 도구를 사용하여 사번을 기반으로 조회합니다.
2.  **내부 정책 수치**: 요청하신 대로 고정된 정책 수치를 안내드립니다.
3.  **상대 수준**: 팀원 평균 데이터가 필요하므로, 사번을 알려주시면 해당 정보를 바탕으로 추정하여 안내드리겠습니다.

**번거로우시겠지만, 사번을 다시 한번 말씀해 주시거나, 제가 조회할 사번을 지정해 주시면 바로 처리해 드리겠습니다.**


---

## 실습 과제: 사내 업무 에이전트 구현

**요구사항**

1. 최소 3개의 `@tool`을 정의합니다 (예: 회의실 예약 조회, 구내식당 메뉴, 출장 승인 상태 등).
2. `create_agent`와 `MemorySaver`로 에이전트를 구성합니다.
3. 동일한 `thread_id`로 3회 이상 대화를 이어가며, 메모리가 정상 동작하는지 확인합니다.
4. 한 가지 질문에서는 반드시 **두 개 이상의 도구를 연쇄 호출**하도록 구성합니다.

In [16]:
# 여기에 코드를 작성하세요
# Hint:
# @tool
# def book_room(room: str, time: str) -> dict: ...
# @tool
# def get_lunch_menu(date: str) -> list[str]: ...
#
# my_agent = create_agent(model=llm, tools=[...], system_prompt="...", checkpointer=MemorySaver())
# cfg = {"configurable": {"thread_id": "my-session"}}
# my_agent.invoke({"messages": [...]}, config=cfg)


<details>
<summary>모범 답안 보기</summary>

```python
@tool
def book_room(room: str, time: str) -> dict:
    """회의실 예약을 시도합니다."""
    return {"status": "booked", "room": room, "time": time}

@tool
def get_lunch_menu(date: str) -> list[str]:
    """구내식당 점심 메뉴 조회 (하드코딩)."""
    return ["불고기 정식", "카레라이스", "비빔국수", "샐러드바"]

@tool
def check_trip_approval(trip_id: str) -> dict:
    """출장 승인 상태를 조회합니다."""
    db = {"T-001": "approved", "T-002": "pending"}
    return {"trip_id": trip_id, "status": db.get(trip_id, "unknown")}

my_tools = [book_room, get_lunch_menu, check_trip_approval]

my_agent = create_agent(
    model=llm,
    tools=my_tools,
    system_prompt="당신은 ABC 사내 업무 어시스턴트입니다. 필요한 도구를 자유롭게 조합해 사용자를 도우세요.",
    checkpointer=MemorySaver(),
)

cfg = {"configurable": {"thread_id": "exercise-001"}}

turns = [
    "내일 10시에 회의실 A 예약해줘",
    "그리고 그날 점심 메뉴도 알려줘",
    "마지막으로 출장 T-001 승인 상태 확인해줘",
]
for t in turns:
    r = my_agent.invoke({"messages": [{"role": "user", "content": t}]}, config=cfg)
    print(">", t)
    print("<", r["messages"][-1].content)
    print()
```

</details>

---

## 보너스 — 미니 여행 플래너 에이전트

다중 도구의 자율 조합을 실제 시나리오로 확인합니다. 항공권, 숙소, 환율 도구를 정의한 뒤, "도쿄 3박 4일" 같은 자연어 요청을 처리하도록 합니다.

In [17]:
@tool
def search_flight(origin: str, destination: str, date: str) -> list[dict]:
    """항공편을 검색합니다. 가격은 만원 단위입니다."""
    return [
        {"airline": "대한항공", "depart": "10:30", "arrive": "12:45", "price_man": 38},
        {"airline": "ANA", "depart": "13:15", "arrive": "15:25", "price_man": 32},
        {"airline": "Peach", "depart": "07:50", "arrive": "10:10", "price_man": 19},
    ]

@tool
def search_hotel(city: str, nights: int, max_price_per_night_man: int) -> list[dict]:
    """도시별 호텔을 검색합니다 (시뮬레이션)."""
    db = {
        "도쿄": [
            {"name": "신주쿠 비즈니스", "price_man": 12, "rating": 4.2},
            {"name": "시부야 모던", "price_man": 18, "rating": 4.5},
            {"name": "아사쿠사 료칸", "price_man": 25, "rating": 4.8},
        ],
        "오사카": [
            {"name": "난바 비즈니스", "price_man": 9, "rating": 4.0},
        ],
    }
    return [h for h in db.get(city, []) if h["price_man"] <= max_price_per_night_man]

@tool
def convert_jpy_to_krw(yen: int) -> dict:
    """엔(JPY) → 원(KRW) 환율 변환 (간단 시뮬레이션)."""
    return {"yen": yen, "krw": int(yen * 9.15)}

travel_agent = create_agent(
    model=llm,
    tools=[search_flight, search_hotel, convert_jpy_to_krw],
    system_prompt=(
        "당신은 여행 플래너입니다. 사용자의 예산·일정에 맞춰 항공/숙소를 추천하고, "
        "총 예상 경비를 만원 단위로 정리해 답하세요. 가성비와 평점을 함께 고려하세요."
    ),
)

q = "서울에서 도쿄로 2026년 5월 10일 출발해서 3박 4일 여행 갈 예정이야. 1인 80만원 예산으로 항공권 + 호텔 추천해줘."
result = travel_agent.invoke({"messages": [{"role": "user", "content": q}]})
print(result["messages"][-1].content)

**✅ 숙소 추천 (가성비 & 평점 고려):**

| 호텔명 | 1박 예상가 (만원) | 총 예상가 (3박) | 평점 | 추천 이유 |
| :--- | :--- | :--- | :--- | :--- |
| **신주쿠 비즈니스** | 12만원 | 36만원 | 4.2 | **최고의 가성비.** 예산에 여유가 생겨 여행 경비에 더 쓸 수 있습니다. |
| **시부야 모던** | 18만원 | 54만원 | 4.5 | **균형 잡힌 선택.** 평점과 가격의 밸런스가 좋고, 시부야 접근성이 좋습니다. |
| 아사쿠사 료칸 | 25만원 | 75만원 | 4.8 | **최고의 만족도.** 평점은 가장 높지만, 예산 초과 위험이 있습니다. |

---

### 💰 최종 추천 조합 및 예상 경비 정리

고객님의 **총 예산 80만원**을 초과하지 않으면서, 만족도 높은 조합을 두 가지로 제안합니다.

#### 🥇 조합 1: 가성비 & 여유 자금 확보 (추천!)
*   **항공권:** Peach (19만원)
*   **숙소:** 신주쿠 비즈니스 (12만원/박 $\times$ 3박 = 36만원)
*   **총 예상 경비:** 19만원 + 36만원 = **55만원**
*   **💰 남은 예산:** 80만원 - 55만원 = **25만원** (식비, 관광지 입장료 등 활동비로 사용 가능)

#### 🥈 조합 2: 안정성 & 만족도 중시 (균형 잡힌 선택)
*   **항공권:** ANA (32만원)
*   **숙소:** 시부야 모던 (18만원/박 $\times$ 3박 = 54만원)
*   **총 예상 경비:** 32만원 + 54만원 = **86만원**
*   **⚠️ 주의:** 이 조합은 **예산(80만원)을 6만원 초과**합니다. 만약 예산을 조금 더 늘릴 수 있다면 가장 만족도가 높을 조합입니다.

---

### ✨ 최종 결론 및 제안

**가장 추천하는 조합은 🥇 조합 1**입니다.

1.  **예산 준수:** 총 예상 경비가 55만원으로, 80만원 예산에 여유롭게 맞습니다.
2.  

---

## 트러블슈팅

| 증상 | 원인 | 해결 방법 |
|---|---|---|
| `ImportError: create_agent` | LangChain v0.x 사용 중 | `pip install -U langchain langgraph` (1.0 이상) |
| 에이전트가 무한 루프 | 모델이 동일 도구를 반복 호출 | `system_prompt`에 "이미 호출한 도구 결과를 그대로 사용"을 명시 |
| 메모리가 동작하지 않음 | `thread_id` 누락 | `config={"configurable": {"thread_id": "..."}}` 전달 필요 |
| `response_format` 미동작 | 모델이 네이티브 구조화 출력 미지원 | `ToolStrategy(Schema)` 명시적 사용 |
| Ollama 로컬 모델의 도구 호출 실패 | 일부 로컬 모델의 도구 호출 정확도 부족 | Gemini 또는 Groq 클라우드 모델로 전환 |
| `@dynamic_prompt`와 `system_prompt` 충돌 | 두 옵션 동시 사용 시 미들웨어가 우선 적용 | 둘 중 하나만 사용 |

---

## 마무리

본 노트북에서 학습한 내용은 다음과 같습니다.

- `create_agent` 단일 호출로 ReAct 에이전트 구성 (04번의 30줄 → 1줄로 축약)
- ReAct 루프 다이어그램 기반의 처리 흐름 이해 (Reason → Act → Observe)
- `agent.stream()`을 통한 사고 과정 실시간 관찰
- `MemorySaver`와 `thread_id`를 활용한 멀티턴 메모리 관리
- `response_format`(`ToolStrategy`, `ProviderStrategy`, 자동 선택) 기반의 구조화 출력
- `@dynamic_prompt` 미들웨어를 활용한 컨텍스트 기반 프롬프트 분기
- 미니 여행 플래너 에이전트 구현 사례

### 다음 노트북(06번) 예고

여기까지의 에이전트는 외부 도구(API)에 접근할 수 있었습니다. 그러나 사내 정책 PDF, 내부 매뉴얼과 같은 **비공개 문서**의 내용은 모델이 알지 못합니다.

다음 노트북에서는 **RAG(Retrieval-Augmented Generation)** 를 다룹니다. 문서를 분할하고 임베딩으로 의미 좌표계에 매핑한 뒤, 질의와 관련도가 높은 문서 조각만 모델에 컨텍스트로 제공하는 방법을 학습합니다.